# Environment configuration — gold/cas_admin
Credentials are external. Set PIPELINE_ENV and job variables generated from config/environments and config/secrets.
This repository revision changes configuration handling, not the source ingestion/write algorithms.


In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession,DataFrame
from dbnd import *
from pyspark.sql import functions as F
from pyspark.sql.types import *
import logging
import time
import os 
import json
import ast
import xml.etree.ElementTree as ET
import traceback
from datetime import datetime, timedelta
import mysql.connector
import base64
import psycopg2
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
# Log Setting
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True 
)
logger = logging.getLogger(__name__)

In [ ]:
# Environment configuration is injected by CP4D/job environment variables.
# Generate a job env file with scripts/build_job_env.py. Never embed credentials here.
def _need_env(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu biến môi trường: " + name)
    return value

PIPELINE_ENV = _need_env("PIPELINE_ENV")
if PIPELINE_ENV not in ("dev", "uat", "prod"):
    raise ValueError("PIPELINE_ENV phải là dev, uat hoặc prod")

# 1. Lấy các biến môi trường
now = datetime.now() + timedelta(hours=7)
yesterday = now - timedelta(days=1)

# 3. Định dạng lại thành chuỗi YYYYMMDD
yesterday = yesterday.strftime("%Y-%m-%d")

#S3
S3_ENDPOINT = _need_env("S3_ENDPOINT")
ACCESS_KEY = _need_env("ACCESS_KEY")
SECRET_KEY = _need_env("SECRET_KEY")
BUCKET_NAME = _need_env("BUCKET_NAME")

#Metastore
WXD_HMS_USERNAME = _need_env("WXD_HMS_USERNAME")
WXD_HMS_PASSWORD = _need_env("WXD_HMS_PASSWORD")
ICEBERG_CATALOG1 = os.getenv("ICEBERG_CATALOG1", "silver")
BUCKET_NAME1 = _need_env("BUCKET_NAME1")
ACCESS_KEY1 = os.getenv("ACCESS_KEY1") or ACCESS_KEY
SECRET_KEY1 = os.getenv("SECRET_KEY1") or SECRET_KEY

HIVEMETASTORE_HOST = _need_env("HIVEMETASTORE_HOST")

HIVE_TRUSTSTORE_PATH = _need_env("HIVE_TRUSTSTORE_PATH")
HIVE_TRUSTSTORE_PASSWORD = _need_env("HIVE_TRUSTSTORE_PASSWORD")

WXD_INSTANCE_ID = _need_env("WXD_INSTANCE_ID")
HADOOP_AMS_URL = _need_env("HADOOP_AMS_URL")
HADOOP_CAS_ENDPOINT = _need_env("HADOOP_CAS_ENDPOINT")
HADOOP_CPG_ENDPOINT = _need_env("HADOOP_CPG_ENDPOINT")


# TABLE_NAME=os.getenv("TABLE_NAME","gold.dwh.dim_org")
# SOURCE_NAME=os.getenv("TABLE_NAME","bronze.crm.emails")
STEP='silver_2_gold'


# ETL_Date = os.getenv("ETL_Date",yesterday)

In [ ]:
TODAY_VN = datetime.now().date()
ETL_DATE = os.getenv("ETL_Date", (TODAY_VN - timedelta(days=1)).isoformat())
RUN_MODE = os.getenv("RUN_MODE", "INCREMENTAL").upper().strip()
TARGET_TABLE = os.getenv("TARGET_TABLE", "gold.dwh.dim_org")
LOG_TABLE = os.getenv("LOG_TABLE", "bronze.pipeline.log_pipeline")
STEP='silver_2_gold'

TRACKED_COLUMNS = [
    "UNIT_NAME", "UNIT_TYPE", "TYPE_CODE", "PARENT_CODE", "COMMUNE_CODE",
    "POSTAL_CODE", "POST_TYPE", "ADDRESS", "STATUS", "TEL",
    "DISTRICT_CODE", "PROVINCE_CODE"
]

In [ ]:
def init_spark():
    spark_builder = SparkSession.builder \
        .appName("silver_2_gold_dim_org") \
        .config(
            "spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
        ) \
        .config(
            "spark.serializer",
            "org.apache.spark.serializer.KryoSerializer"
        ) \
        .config("spark.hive.metastore.uris", HIVEMETASTORE_HOST) \
        .config("spark.hive.metastore.use.SSL", "true") \
        .config("spark.hive.metastore.truststore.type", "JKS") \
        .config("spark.hive.metastore.truststore.path", HIVE_TRUSTSTORE_PATH) \
        .config("spark.hive.metastore.truststore.password", HIVE_TRUSTSTORE_PASSWORD) \
        .config("spark.hive.metastore.client.auth.mode", "PLAIN") \
        .config("spark.hive.metastore.client.plain.username", WXD_HMS_USERNAME) \
        .config("spark.hive.metastore.client.plain.password", WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.client.thrift.transport.mode", "http") \
        .config("spark.hadoop.wxd.ams.url", HADOOP_AMS_URL) \
        .config("spark.hadoop.wxd.cas.endpoint", HADOOP_CAS_ENDPOINT) \
        .config("spark.hadoop.wxd.cpg.endpoint", HADOOP_CPG_ENDPOINT) \
        .config("spark.hadoop.wxd.instanceId", WXD_INSTANCE_ID) \
        .config(
            "spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem"
        ) \
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.endpoint",
            S3_ENDPOINT
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.access.key",
            ACCESS_KEY
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.secret.key",
            SECRET_KEY
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.path.style.access",
            "true"
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.endpoint",
            S3_ENDPOINT
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.access.key",
            ACCESS_KEY1
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.secret.key",
            SECRET_KEY1
        ) \
        .config(
            f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.path.style.access",
            "true"
        ) \
        .config("spark.hadoop.iceberg.engine.hive.lock-enabled", "false") \
        .config("spark.sql.catalogImplementation", "hive")

    # Kết nối đồng thời 3 Iceberg Catalog
    for catalog in ["bronze", "silver", "gold"]:
        spark_builder = spark_builder \
            .config(
                f"spark.sql.catalog.{catalog}",
                "org.apache.iceberg.spark.SparkCatalog"
            ) \
            .config(
                f"spark.sql.catalog.{catalog}.type",
                "hive"
            ) \
            .config(
                f"spark.sql.catalog.{catalog}.uri",
                f"{HIVEMETASTORE_HOST}?catalog={catalog}"
            ) \
            .config(
                f"spark.sql.catalog.{catalog}.warehouse",
                f"s3a://{BUCKET_NAME1}"
            ) \
            .config(
                f"spark.sql.catalog.{catalog}.hive.metastore.client.plain.username",
                WXD_HMS_USERNAME
            ) \
            .config(
                f"spark.sql.catalog.{catalog}.hive.metastore.client.plain.password",
                WXD_HMS_PASSWORD
            )

    spark = spark_builder.getOrCreate()

    spark.sparkContext.setLogLevel("WARN")

    return spark

In [ ]:
start_date = datetime.now()
log_message = None
status = "Success"
record_count = 0
spark = None

TODAY_VN = datetime.now().date()
ETL_DATE = os.getenv("ETL_Date", (TODAY_VN - timedelta(days=1)).isoformat())
LOAD_DATE = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y-%m-%d")
# RUN_MODE = os.getenv("RUN_MODE", "INCREMENTAL").upper().strip()
TABLE_NAME = os.getenv("TABLE_NAME", "gold.dwh.dim_org")
LOG_TABLE = os.getenv("LOG_TABLE", "bronze.pipeline.log_pipeline")
TABLE_NAME = TARGET_TABLE
STEP = "silver_2_gold"

In [ ]:
# ------------------------------------------------------------------
# 1. KHAI BÁO CÁC CÂU LỆNH SQL (chưa thực thi)
# ------------------------------------------------------------------

SQL_ADD_DATA_DATE = f"""
ALTER TABLE {TABLE_NAME}
ADD COLUMNS (data_date DATE)
"""

SQL_ADD_INGEST_TIME = f"""
ALTER TABLE {TABLE_NAME}
ADD COLUMNS (ingest_time TIMESTAMP)
"""

SQL_ADD_BUSINESS_HASH = f"""
ALTER TABLE {TABLE_NAME}
ADD COLUMNS (business_hash STRING)
"""


SQL_CREATE_TABLE = f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    ORG_SKEY BIGINT,
    UNIT_CODE STRING,
    UNIT_NAME STRING, UNIT_TYPE STRING, TYPE_CODE INT,
    PARENT_CODE STRING, COMMUNE_CODE STRING, POSTAL_CODE STRING,
    POST_TYPE STRING, ADDRESS STRING, STATUS INT, TEL STRING,
    DISTRICT_CODE STRING, PROVINCE_CODE STRING,
    F_LATEST_RECORD_INDICATOR STRING,
    RECORD_START_DATE DATE, RECORD_END_DATE DATE,
    REC_CREATE_DATE TIMESTAMP, data_date DATE,
    ingest_time TIMESTAMP, business_hash STRING
) USING iceberg
"""

SQL_SOURCE = f"""
CREATE OR REPLACE TEMP VIEW v_org_source AS
WITH src AS (
    SELECT
        TRIM(CAST(UNIT_CODE AS STRING)) AS UNIT_CODE,
        CAST(UNIT_NAME AS STRING) AS UNIT_NAME,
        CAST(UNIT_TYPE AS STRING) AS UNIT_TYPE,
        CAST(TYPE_CODE AS INT) AS TYPE_CODE,
        CAST(PARENT_CODE AS STRING) AS PARENT_CODE,
        CAST(COMMUNE_CODE AS STRING) AS COMMUNE_CODE,
        CAST(POSTAL_CODE AS STRING) AS POSTAL_CODE,
        CAST(POST_TYPE AS STRING) AS POST_TYPE,
        CAST(ADDRESS AS STRING) AS ADDRESS,
        CAST(STATUS AS INT) AS STATUS,
        CAST(TEL AS STRING) AS TEL,
        CAST(DISTRICT_CODE AS STRING) AS DISTRICT_CODE,
        CAST(PROVINCE_CODE AS STRING) AS PROVINCE_CODE,
        CAST(data_date AS DATE) AS data_date,
        CAST(ingest_time AS TIMESTAMP) AS ingest_time
    FROM silver.cas_admin.mcas_organization_standard
    WHERE ingest_time >= TIMESTAMP '{LOAD_DATE} 00:00:00'
      AND ingest_time < CAST(
          DATE_ADD(DATE '{LOAD_DATE}', 1) AS TIMESTAMP
      )
)
SELECT *,
       SHA2(TO_JSON(NAMED_STRUCT(
           'UNIT_NAME', UNIT_NAME,
           'UNIT_TYPE', UNIT_TYPE,
           'TYPE_CODE', TYPE_CODE,
           'PARENT_CODE', PARENT_CODE,
           'COMMUNE_CODE', COMMUNE_CODE,
           'POSTAL_CODE', POSTAL_CODE,
           'POST_TYPE', POST_TYPE,
           'ADDRESS', ADDRESS,
           'STATUS', STATUS,
           'TEL', TEL,
           'DISTRICT_CODE', DISTRICT_CODE,
           'PROVINCE_CODE', PROVINCE_CODE
       ), MAP('ignoreNullFields', 'false')), 256) AS business_hash
FROM src
"""

SQL_INVALID = """
SELECT COUNT(*) AS cnt
FROM v_org_source
WHERE UNIT_CODE IS NULL OR UNIT_CODE = ''
   OR data_date IS NULL OR ingest_time IS NULL OR STATUS IS NULL
"""

SQL_DUPLICATES = """
SELECT COUNT(*) AS cnt
FROM (
    SELECT UNIT_CODE
    FROM v_org_source
    GROUP BY UNIT_CODE
    HAVING COUNT(*) > 1
) x
"""

SQL_CHANGES = f"""
CREATE OR REPLACE TEMP VIEW v_org_changes AS
SELECT s.*,
       t.ORG_SKEY AS old_skey,
       t.RECORD_START_DATE AS old_start_date
FROM v_org_source s
LEFT JOIN {TABLE_NAME} t
  ON s.UNIT_CODE = t.UNIT_CODE
 AND t.F_LATEST_RECORD_INDICATOR = '1'
WHERE t.UNIT_CODE IS NULL
   OR NOT (s.business_hash <=> t.business_hash)
"""

SQL_LATE = """
SELECT COUNT(*) AS cnt
FROM v_org_changes
WHERE old_skey IS NOT NULL AND data_date <= old_start_date
"""

SQL_COUNT = "SELECT COUNT(*) AS cnt FROM v_org_changes"

SQL_NEW = f"""
CREATE OR REPLACE TEMP VIEW v_org_new AS
SELECT s.*,
       (SELECT COALESCE(MAX(ORG_SKEY), 0) FROM {TABLE_NAME})
           + ROW_NUMBER() OVER (ORDER BY UNIT_CODE) AS ORG_SKEY
FROM v_org_changes s
"""

SQL_STAGE = """
CREATE OR REPLACE TEMP VIEW v_org_stage AS
SELECT s.*, s.UNIT_CODE AS merge_key, 'CLOSE' AS op
FROM v_org_new s
WHERE old_skey IS NOT NULL
UNION ALL
SELECT s.*, CAST(NULL AS STRING) AS merge_key, 'INSERT' AS op
FROM v_org_new s
"""


SQL_MERGE = f"""
MERGE INTO {TABLE_NAME} t
USING v_org_stage s
ON t.UNIT_CODE = s.merge_key
AND t.F_LATEST_RECORD_INDICATOR = '1'

WHEN MATCHED AND s.op = 'CLOSE'
THEN UPDATE SET
    t.F_LATEST_RECORD_INDICATOR = '0',
    t.RECORD_END_DATE = DATE_SUB(s.data_date, 1)

WHEN NOT MATCHED AND s.op = 'INSERT'
THEN INSERT (
    ORG_SKEY, UNIT_CODE, UNIT_NAME, UNIT_TYPE, TYPE_CODE,
    PARENT_CODE, COMMUNE_CODE, POSTAL_CODE, POST_TYPE,
    ADDRESS, STATUS, TEL, DISTRICT_CODE, PROVINCE_CODE,
    F_LATEST_RECORD_INDICATOR, RECORD_START_DATE,
    RECORD_END_DATE, REC_CREATE_DATE, data_date,
    ingest_time, business_hash
)
VALUES (
    s.ORG_SKEY, s.UNIT_CODE, s.UNIT_NAME, s.UNIT_TYPE,
    s.TYPE_CODE, s.PARENT_CODE, s.COMMUNE_CODE,
    s.POSTAL_CODE, s.POST_TYPE, s.ADDRESS, s.STATUS,
    s.TEL, s.DISTRICT_CODE, s.PROVINCE_CODE,
    '1',
    s.data_date,
    CAST(NULL AS DATE),
    CURRENT_TIMESTAMP(),
    s.data_date,
    s.ingest_time,
    s.business_hash
)
"""




In [ ]:
# ------------------------------------------------------------------
# 2. THỰC THI SQL THEO THỨ TỰ + GHI LOG NHƯ CODE BASE
# ------------------------------------------------------------------
try:

    spark = init_spark()  # Hàm init_spark() giữ nguyên từ code base.
    spark.sql(SQL_CREATE_TABLE)

    # 2. Kiểm tra các cột hiện có
    existing_columns = [
        col.lower() for col in spark.table(TABLE_NAME).columns
    ]
    
    # 3. Chỉ thêm cột nếu chưa tồn tại
    if "data_date" not in existing_columns:
        spark.sql(SQL_ADD_DATA_DATE)
    
    if "ingest_time" not in existing_columns:
        spark.sql(SQL_ADD_INGEST_TIME)
    
    if "business_hash" not in existing_columns:
        spark.sql(SQL_ADD_BUSINESS_HASH)

    logger.info(
        f"Đang lấy dữ liệu bảng silver.cas_admin.mcas_organization_standard "
        f"với ingest_time thuộc ngày {LOAD_DATE}."
    )

    spark.sql(SQL_SOURCE)

    logger.info(
        f"Đã lấy dữ liệu bảng silver.cas_admin.mcas_organization_standard "
        f"với ingest_time thuộc ngày {LOAD_DATE}."
    )

    invalid = spark.sql(SQL_INVALID).first()[0]
    duplicates = spark.sql(SQL_DUPLICATES).first()[0]
    if invalid or duplicates:
        raise RuntimeError(
            f"Silver không hợp lệ: {invalid} dòng lỗi, {duplicates} khóa trùng"
        )

    spark.sql(SQL_CHANGES)
    late = spark.sql(SQL_LATE).first()[0]
    if late:
        raise RuntimeError(
            f"{late} khóa có thay đổi cùng ngày/cũ hơn phiên bản Gold hiện hành"
        )

    record_count = spark.sql(SQL_COUNT).first()[0]
    spark.sql(SQL_NEW)
    spark.sql(SQL_STAGE)
    spark.sql(SQL_MERGE)

    log_message = f"D={ETL_DATE}; SCD2_CHANGED={record_count}"
    logger.info(f"DIM_ORG SCD2 hoàn tất: {log_message}")

except Exception as e:
    log_message = (
        f"Error Type: {type(e).__name__} - Message: {str(e)}\n"
        f"{traceback.format_exc()}"
    )
    status = "Fail"
    raise

finally:
    # record_count đã lấy trước MERGE, không gọi df.count() sau MERGE.
    if status.upper() == "SUCCESS" and record_count == 0:
        logger.info(f"Không có thay đổi DIM_ORG ngày {ETL_DATE}. Bỏ qua ghi log.")
    elif spark is not None:
        end_date = datetime.now()
        schema_log_table = StructType([
            StructField("target_table", StringType(), True),
            StructField("source_table", StringType(), True),
            StructField("log_message", StringType(), True),
            StructField("status", StringType(), True),
            StructField("start_date", TimestampType(), True),
            StructField("end_date", TimestampType(), True),
            StructField("step", StringType(), True),
            StructField("record_count", LongType(), True),
        ])

        data_log = [(
            TABLE_NAME,
            None,
            log_message,
            status,
            start_date,
            end_date,
            STEP,
            record_count,
        )]

        df_log = spark.createDataFrame(data_log, schema=schema_log_table)
        (
            df_log.write
                .option("write.target-file-size-bytes", "134217728")
                .option("write.distribution-mode", "none")
                .format("iceberg")
                .mode("append")
                .saveAsTable(LOG_TABLE)
        )
        logger.info("Ghi log thành công.")
    else:
        logger.error("Không thể ghi log vì Spark chưa khởi tạo thành công.")


In [ ]:
spark.sql(f"""
    SELECT
        target_table,
        status,
        log_message,
        start_date,
        end_date,
        step,
        record_count
    FROM {LOG_TABLE}
    WHERE target_table = '{TABLE_NAME}'
      AND step = '{STEP}'
      AND log_message LIKE 'D={ETL_DATE};%'
    ORDER BY start_date DESC
    LIMIT 10
""").show(truncate=False)